In [ ]:
# Run once to download
!wget https://raw.githubusercontent.com/jurgjn/mgen-pairs-pools/refs/heads/main/mgen_pairwise.tsv

In [ ]:
import numpy as np, pandas as pd, sklearn, matplotlib, matplotlib.pyplot as plt, seaborn as sns, pooled_ppi

In [ ]:
pairs = pd.read_csv('mgen_pairwise.tsv', sep='\t')
pairs['pair_iptm_mean_corrected_recap'] = pooled_ppi.size_correction.size_correction(pairs['pair_tokens'], pairs['pair_iptm_mean'])
pairs

In [ ]:
pairs = pairs.groupby(['group', 'af3_id1', 'af3_id2']).head(1).reset_index()
pairs['string_experimental_800'] = pairs['string_experimental'] > 800
pairs['string_experimental_800'].value_counts()

In [ ]:
def auc_(group, labels_col_='string_experimental_800', scores_col_='pair_iptm_mean'):
    pairs_subset_ = pairs.query('group == @group')
    labels_ = pairs_subset_[labels_col_].tolist()
    scores_ = pairs_subset_[scores_col_].tolist()
    auc_ = sklearn.metrics.roc_auc_score(y_true=labels_, y_score=scores_)
    return group, sum(pairs_subset_[labels_col_]), sum(~pairs_subset_[labels_col_]), auc_, scores_col_

aucs_ = pd.DataFrame.from_records([
    auc_("pairs", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean'),
    auc_("pools_2k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean'),
    auc_("pools_3k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean'),
    auc_("pools_4k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean'),
    auc_("pools_5k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean'),
    auc_("pairs", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected'),
    auc_("pools_2k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected'),
    auc_("pools_3k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected'),
    auc_("pools_4k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected'),
    auc_("pools_5k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected'),
    auc_("pairs", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected_recap'),
    auc_("pools_2k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected_recap'),
    auc_("pools_3k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected_recap'),
    auc_("pools_4k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected_recap'),
    auc_("pools_5k", labels_col_='string_experimental_800', scores_col_='pair_iptm_mean_corrected_recap'),
], columns=['group', 'n_pos', 'n_neg', 'auc', 'metric'])
aucs_

In [ ]:
# Re-plot MGen pools-pairs data with original size-correction, and recap with default coefficients
plt.figure(figsize=(4, 4))
sns.barplot(aucs_, x='auc', y='group', hue='metric')
plt.legend(loc=(.7, .76))